In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    base_url=os.getenv("IQ_BASE_URL")
)

In [2]:
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x10dcf4950>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10e6ea690>, root_client=<openai.OpenAI object at 0x10d942290>, root_async_client=<openai.AsyncOpenAI object at 0x10d957750>, model_name='gpt-5-chat', openai_api_key=SecretStr('**********'), openai_api_base='https://api.iq.cudasvc.com/', openai_proxy='')

### **Defining Structure**

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="The year the movie got released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movie rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x10dcf4950>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10e6ea690>, root_client=<openai.OpenAI object at 0x10d942290>, root_async_client=<openai.AsyncOpenAI object at 0x10d957750>, model_name='gpt-5-chat', openai_api_key=SecretStr('**********'), openai_api_base='https://api.iq.cudasvc.com/', openai_proxy=''), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie got released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'parallel_tool_calls': False, 'tool_cho

In [5]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='Sure! Here’s a detailed overview of *Inception*:\n\n**Title:** *Inception*  \n**Release Date:** July 16, 2010  \n**Director & Writer:** Christopher Nolan  \n**Genre:** Science fiction, action, thriller  \n**Runtime:** Approximately 148 minutes  \n**Language:** English  \n**Country:** United States / United Kingdom  \n\n---\n\n### **Synopsis**\n*Inception* is a mind-bending science fiction thriller centered around the concept of shared dreaming and extracting or implanting ideas in a person\'s subconscious. The story follows **Dom Cobb** (played by Leonardo DiCaprio), a skilled "extractor" who specializes in stealing secrets from deep within a person\'s mind during the dream state. Cobb is offered a chance to have his criminal record erased if he can successfully perform *inception*—planting an idea into someone’s mind—on the heir to a corporate empire.\n\n---\n\n### **Main Cast**\n- **Leonardo DiCaprio** as Dom Cobb  \n- **Joseph Gordon-Levitt** as Arthur  \n- **Elli

In [6]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
# To get both raw and structured output
model_with_structure_and_raw = model.with_structured_output(Movie, include_raw=True)
model_with_structure_and_raw.invoke("Provide details about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yMjV5JJ9vrBL0SbataDnrsAn', 'function': {'arguments': '{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', 'name': 'Movie'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 106, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-chat', 'system_fingerprint': 'fp_1133e9eff6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-2252891e-a6e5-4122-a3fe-076d38964475-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'call_yMjV5JJ9vrBL0SbataDnrsAn', 'type': 'tool_call'}], usage_metadata={'input_tokens': 106, 'output_tokens': 23, 'total_token

### **Nested Structure**

In [10]:
class Actor(BaseModel):
    name: str 
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
model_with_structure.invoke("Provide details about the movie Inception")

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)